In [2]:
#!/usr/bin/env python3
import pandas as pd
from pathlib import Path
from glob import glob
from collections import defaultdict
import numpy as np

# 1. 排除 Breast 相关 Slide
EXCLUDE_TISSUES = {
    "Xenium_V1_FFPE_Human_Breast_IDC_With_Addon",
    "Xenium_V1_FFPE_Human_Breast_ILC",
    "Xenium_V1_FFPE_Human_Breast_IDC",
    "Xenium_V1_FFPE_Human_Breast_ILC_With_Addon",
    "Xenium_V1_FFPE_Human_Breast_IDC_Big_1",
    "Xenium_V1_FFPE_Human_Breast_IDC_Big_2",
}

# 2. 排除低频及噪声 Label (21类以外的)
EXCLUDE_LABELS = {
    "Breast cancer cells",
    "Adipocytes",
    "Cardiac muscle cells",
    "OPCs",
    "Brain cancer cells",
    "Kidney cancer cells",
    "Erythrocytes",
}

# 数据目录
DATA_DIR = "/hpc/group/jilab/rz179/MorphPT_MOE/prepared/shards_multiview_parquet"

def main():
    pattern = f"{DATA_DIR}/*.parquet"
    files = sorted(glob(pattern))
    
    if not files:
        pattern = f"{DATA_DIR}/tissue=*/part.parquet"
        files = sorted(glob(pattern))
        
    stats = []
    print(f"Scanning {len(files)} shards in {DATA_DIR}...")

    for f in files:
        if "tissue=" in f:
            tissue = Path(f).parent.name.replace("tissue=", "")
        else:
            tissue = Path(f).stem
            
        if tissue in EXCLUDE_TISSUES:
            continue
            
        try:
            df = pd.read_parquet(f, columns=["label"])
            df = df[~df["label"].isin(EXCLUDE_LABELS)]
            
            counts = df["label"].value_counts().to_dict()
            for label, count in counts.items():
                stats.append({
                    "tissue": tissue,
                    "label": label,
                    "count": count
                })
        except Exception as e:
            print(f"Error reading {f}: {e}")
            
    df_stats = pd.DataFrame(stats)
    
    if df_stats.empty:
        print("Error: No data found.")
        return

    # --- 分析汇总 ---
    print("\n" + "="*95)
    print("SLIDE-LEVEL DISTRIBUTION ANALYSIS (v4 Dataset: 21 Classes)")
    print("="*95)
    
    thresholds = [200, 500, 1000]
    summary = []
    
    for label in sorted(df_stats["label"].unique()):
        class_data = df_stats[df_stats["label"] == label]
        n_slides = len(class_data)
        total_count = class_data["count"].sum()  # 增加总数统计
        avg_count = class_data["count"].mean()
        min_count = class_data["count"].min()
        
        row = {
            "Fine Class": label[:25],
            "Total": int(total_count),           # 显式列出总数
            "Slides": n_slides,
            "Avg": int(avg_count),
            "Min": int(min_count)
        }
        
        for t in thresholds:
            short_count = len(class_data[class_data["count"] < t])
            row[f"< {t}"] = f"{short_count} ({short_count/n_slides:.0%})"
            
        summary.append(row)
        
    df_summary = pd.DataFrame(summary)
    # 按总数 Total 降序排列，方便看整体规模
    df_summary = df_summary.sort_values("Total", ascending=False)
    
    print(df_summary.to_string(index=False))
    
    print("\n" + "="*95)
    print(f"Grand Total Cells: {df_stats['count'].sum():,}")
    print(f"Total Unique Slides: {df_stats['tissue'].nunique()}")
    print("="*95)

if __name__ == "__main__":
    main()

Scanning 34 shards in /hpc/group/jilab/rz179/MorphPT_MOE/prepared/shards_multiview_parquet...

SLIDE-LEVEL DISTRIBUTION ANALYSIS (v4 Dataset: 21 Classes)
               Fine Class   Total  Slides    Avg    Min  < 200   < 500  < 1000
                  T cells 1167799      25  46711    226 0 (0%)  1 (4%)  1 (4%)
                  B cells 1046447      18  58135   1052 0 (0%)  0 (0%)  0 (0%)
            Myeloid cells  946694      25  37867    566 0 (0%)  0 (0%)  1 (4%)
         Epithelial cells  673056      22  30593     18 1 (5%)  1 (5%)  1 (5%)
Stem and progenitor cells  626997       5 125399  24176 0 (0%)  0 (0%)  0 (0%)
        Endothelial cells  520721      24  21696   1155 0 (0%)  0 (0%)  0 (0%)
              Fibroblasts  396271      19  20856   1266 0 (0%)  0 (0%)  0 (0%)
      Smooth muscle cells  270506      18  15028   1277 0 (0%)  0 (0%)  0 (0%)
       Ovary cancer cells  190324       1 190324 190324 0 (0%)  0 (0%)  0 (0%)
       Colon cancer cells  176040       2  88020  76208 

In [1]:
import json
from pathlib import Path

SPLITS = Path("/hpc/group/jilab/rz179/MorphPT_MOE/prepared/splits_v2_seed1337_nobreast")

EXPERTS = {
    "Cancer": sorted([
        "Colon cancer cells", "Liver cancer cells", "Lung cancer cells",
        "Ovary cancer cells", "Pancreas cancer cells", "Skin cancer cells",
    ]),
    "Lymphoid": sorted(["T cells", "B cells", "NK cells"]),
    "Neuroglial": sorted(["Astrocytes", "Microglia", "Neurons", "Oligodendrocytes"]),
    "Tissue_Vascular": sorted([
        "Epithelial cells", "Fibroblasts", "Pericytes",
        "Endothelial cells", "Myeloid cells", "Smooth muscle cells",
    ]),
}

for name, classes in EXPERTS.items():
    d = {c: i for i, c in enumerate(classes)}
    out = SPLITS / f"expert_{name}" / "class_to_idx.json"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(d, indent=2))
    print(f"{name}: {len(d)} classes -> {out}")

Cancer: 6 classes -> /hpc/group/jilab/rz179/MorphPT_MOE/prepared/splits_v2_seed1337_nobreast/expert_Cancer/class_to_idx.json
Lymphoid: 3 classes -> /hpc/group/jilab/rz179/MorphPT_MOE/prepared/splits_v2_seed1337_nobreast/expert_Lymphoid/class_to_idx.json
Neuroglial: 4 classes -> /hpc/group/jilab/rz179/MorphPT_MOE/prepared/splits_v2_seed1337_nobreast/expert_Neuroglial/class_to_idx.json
Tissue_Vascular: 6 classes -> /hpc/group/jilab/rz179/MorphPT_MOE/prepared/splits_v2_seed1337_nobreast/expert_Tissue_Vascular/class_to_idx.json
